# Time-Series Forecasting with Prophet and a Seasonal Baseline

**Type:** Original supplementary practice  
**Data:** Synthetic daily observations generated below

This notebook uses Prophet and compares it with a simple weekly seasonal-naive baseline. A chronological holdout is used; the forecast horizon is fixed before model fitting. Synthetic data demonstrate the workflow only and do not validate forecasting performance in a real domain.

Install Prophet using the repository's supplementary requirements before running.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

rng = np.random.default_rng(321)
periods = 900
dates = pd.date_range("2022-01-01", periods=periods, freq="D")
t = np.arange(periods)
weekly = 8*np.sin(2*np.pi*t/7) + 3*np.cos(2*np.pi*t/7)
annual = 14*np.sin(2*np.pi*t/365.25)
trend = 0.018*t
noise = rng.normal(0, 3.5, periods)
data = pd.DataFrame({"ds":dates, "y":95+trend+weekly+annual+noise})

holdout = 90
train = data.iloc[:-holdout].copy()
test = data.iloc[-holdout:].copy()
print("Training:", train["ds"].min().date(), "to", train["ds"].max().date())
print("Test:    ", test["ds"].min().date(), "to", test["ds"].max().date())

## 1. Plot the observed series

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(train["ds"], train["y"], label="Training data", linewidth=1)
ax.plot(test["ds"], test["y"], label="Holdout data", linewidth=1)
ax.set(title="Synthetic daily series", xlabel="Date", ylabel="Value")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 2. Seasonal-naive baseline

For daily data with weekly seasonality, the baseline predicts each day using the value seven days earlier. This is a useful benchmark: a more complex model should be compared against a simple method.

In [ ]:
history = train.set_index("ds")["y"]
# Multi-step seasonal-naive forecast: repeat the final observed weekly pattern.
last_week = train["y"].iloc[-7:].to_numpy()
baseline = np.array([last_week[i % 7] for i in range(len(test))])

def score(actual, predicted, name):
    return {
        "Model":name,
        "MAE":mean_absolute_error(actual, predicted),
        "RMSE":np.sqrt(mean_squared_error(actual, predicted))
    }

print(score(test["y"], baseline, "Weekly seasonal naive"))


## 3. Fit Prophet

Prophet models trend and seasonal components. Its default uncertainty intervals rely on modelling assumptions and should not be treated as guaranteed coverage. Model settings should be chosen based on domain knowledge and validation, not by repeatedly inspecting the final test set.

In [ ]:
prophet = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.90
)
prophet.fit(train)
future = test[["ds"]].copy()
forecast = prophet.predict(future)
pred = forecast["yhat"].to_numpy()

prophet_score = score(test["y"], pred, "Prophet")
results = pd.DataFrame([
    score(test["y"], baseline, "Weekly seasonal naive"),
    prophet_score
]).sort_values("RMSE")
display(results.round(3))

## 4. Visualise forecast and uncertainty

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(train["ds"], train["y"], label="Train", alpha=.7)
ax.plot(test["ds"], test["y"], label="Actual holdout", linewidth=1.6)
ax.plot(forecast["ds"], forecast["yhat"], label="Prophet forecast", linewidth=1.6)
ax.fill_between(
    forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
    alpha=.18, label="90% model interval"
)
ax.set(title="Prophet forecast on synthetic holdout", xlabel="Date", ylabel="Value")
ax.legend()
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## Interpretation and limitations

Compare the model against the baseline, inspect errors over time, and check whether residuals retain seasonal structure. This single chronological split is not a substitute for rolling-origin backtesting. For business use, evaluate multiple forecast origins, investigate holidays and structural breaks, and document the cost of over- and under-forecasting.

**Data caveat:** This dataset is generated by a known synthetic formula. Performance here does not establish that Prophet will outperform other models on real sales, energy or demand data.